In [ ]:
import re
# Import pyologger utilities
from pyologger.utils.folder_manager import *
from pyologger.plot_data.plotter import *
from pyologger.utils.param_manager import ParamManager
from pyologger.load_data.datareader import DataReader
from pyologger.load_data.metadata import Metadata

# Load important file paths and configurations
config, data_dir, color_mapping_path, montage_path = load_configuration()
data_dir
### Fetch metadata

import os
import json
import pickle
import tempfile
from datetime import datetime, timedelta

# ---- user-provided context assumed ----
# data_dir: base data directory
# config: dict with paths (e.g., config["paths"]["local_repo_path"])
# Metadata: the class you just updated

overwrite = False  # Force refresh from Notion if True

# Paths
metadata_dir = os.path.join(data_dir, "00_Metadata")
os.makedirs(metadata_dir, exist_ok=True)
metadata_pickle_path = os.path.join(metadata_dir, "metadata_snapshot.pkl")

relations_map_dir = config["paths"].get("local_repo_path", metadata_dir)
os.makedirs(relations_map_dir, exist_ok=True)
relations_map_path = os.path.join(relations_map_dir, "relations_map.json")


def atomic_write_bytes(path: str, data: bytes):
    """Write bytes atomically to avoid partial/corrupt files."""
    dirpath = os.path.dirname(path)
    os.makedirs(dirpath, exist_ok=True)
    with tempfile.NamedTemporaryFile(dir=dirpath, delete=False) as tmp:
        tmp.write(data)
        tmp.flush()
        os.fsync(tmp.fileno())
        tmp_path = tmp.name
    os.replace(tmp_path, path)


def atomic_write_text(path: str, text: str):
    atomic_write_bytes(path, text.encode("utf-8"))


def load_all_tables(md_obj):
    """
    Convenience: pull all the DB/data source tables from a Metadata instance.
    Returns a dict so you can unpack if you want.
    """
    return {
        "deployment_db": md_obj.get_metadata("deployment_DB"),
        "logger_db": md_obj.get_metadata("logger_DB"),
        "recording_db": md_obj.get_metadata("recording_DB"),
        "animal_db": md_obj.get_metadata("animal_DB"),
        "dataset_db": md_obj.get_metadata("dataset_DB"),
        "procedure_db": md_obj.get_metadata("procedure_DB"),
        "observation_db": md_obj.get_metadata("observation_DB"),
        "collaborator_db": md_obj.get_metadata("collaborator_DB"),
        "location_db": md_obj.get_metadata("location_DB"),
        "montage_db": md_obj.get_metadata("montage_DB"),
        "signal_db": md_obj.get_metadata("signal_DB"),
        "attachment_db": md_obj.get_metadata("attachment_DB"),
        "originalchannel_db": md_obj.get_metadata("originalchannel_DB"),
        "standardizedchannel_db": md_obj.get_metadata("standardizedchannel_DB")
    }


def pickle_needs_refresh(path: str, max_age_days: int = 14) -> bool:
    if not os.path.exists(path):
        return True
    try:
        mtime = datetime.fromtimestamp(os.path.getmtime(path))
        return (datetime.now() - mtime) > timedelta(days=max_age_days)
    except Exception:
        # If anything is weird with the file, refresh.
        return True


# Decide whether to pull fresh data from Notion
needs_refresh = overwrite or pickle_needs_refresh(metadata_pickle_path, max_age_days=14)

if needs_refresh:
    # 1) Build a fresh Metadata instance (hits Notion and populates self.metadata etc.)
    metadata = Metadata()

    # 2) Recompute relations map (uses databases.retrieve schemas and relation fields)
    metadata.map_database_relations()
    relations_map = metadata.relations_map

    # 3) Save relations map atomically (so it’s never half-written)
    try:
        atomic_write_text(relations_map_path, json.dumps(relations_map, indent=4))
        print(f"Relations map saved at: {relations_map_path}")
    except Exception as e:
        print(f"[WARN] Failed to write relations_map.json: {e}")

    # 4) Strip live client / transient runtime caches before pickling
    metadata.notion = None
    if hasattr(metadata, "data_source_cache"):
        metadata.data_source_cache = {}

    # Optional: embed a tiny snapshot header for sanity
    snapshot_meta = {
        "notion_version": getattr(metadata, "notion_version", None),
        "created_at": datetime.now().isoformat(),
        "class": "Metadata",
    }
    payload = {"snapshot_meta": snapshot_meta, "metadata_obj": metadata}

    # 5) Snapshot full metadata object to disk atomically
    try:
        atomic_write_bytes(metadata_pickle_path, pickle.dumps(payload, protocol=pickle.HIGHEST_PROTOCOL))
        print(f"[REFRESH] Metadata snapshot saved at: {metadata_pickle_path}")
    except Exception as e:
        print(f"[ERROR] Failed to write metadata snapshot: {e}")
        raise
else:
    # Load cached snapshot instead of hitting Notion
    print(f"[CACHE] Using existing metadata snapshot at: {metadata_pickle_path}")
    try:
        with open(metadata_pickle_path, "rb") as file:
            payload = pickle.load(file)
        # Backward-compat: support old format (raw Metadata pickled directly)
        if isinstance(payload, dict) and "metadata_obj" in payload:
            metadata = payload["metadata_obj"]
            snapshot_meta = payload.get("snapshot_meta", {})
        else:
            metadata = payload
            snapshot_meta = {}
        # Note: metadata.notion is None here (by design). We're only reading dfs, so it's fine.
    except Exception as e:
        print(f"[WARN] Cache unreadable ({e}). Falling back to fresh pull.")
        metadata = Metadata()
        metadata.map_database_relations()
        relations_map = metadata.relations_map
        try:
            atomic_write_text(relations_map_path, json.dumps(relations_map, indent=4))
        except Exception as ee:
            print(f"[WARN] Failed to write relations_map.json on fallback: {ee}")
        metadata.notion = None
        if hasattr(metadata, "data_source_cache"):
            metadata.data_source_cache = {}
        payload = {
            "snapshot_meta": {
                "notion_version": getattr(metadata, "notion_version", None),
                "created_at": datetime.now().isoformat(),
                "class": "Metadata",
            },
            "metadata_obj": metadata,
        }
        atomic_write_bytes(metadata_pickle_path, pickle.dumps(payload, protocol=pickle.HIGHEST_PROTOCOL))
        print(f"[REFRESH] Metadata snapshot saved at: {metadata_pickle_path}")


# Expose each table for downstream code in this session
tables = load_all_tables(metadata)

deployment_db = tables["deployment_db"]
logger_db = tables["logger_db"]
recording_db = tables["recording_db"]
animal_db = tables["animal_db"]
dataset_db = tables["dataset_db"]
procedure_db = tables["procedure_db"]
observation_db = tables["observation_db"]
collaborator_db = tables["collaborator_db"]
location_db = tables["location_db"]
montage_db = tables["montage_db"]
signal_db = tables["signal_db"]
attachment_db = tables["attachment_db"]
originalchannel_db = tables["originalchannel_db"]
standardizedchannel_db = tables["standardizedchannel_db"]

# Optional: quick sanity print of row counts
try:
    counts = {k: (v.shape[0] if v is not None else 0) for k, v in tables.items()}
    print("[Metadata tables] row counts:", json.dumps(counts, indent=2))
except Exception:
    pass


In [ ]:
import pandas as pd

DEPLOYMENT_KEY = "Deployment ID"
RECORDING_KEY = "Recording ID"
LOGGER_KEY = "Logger ID"   # <-- change if needed (e.g., "LoggerID")

# --- time zones from deployment ---
dep_tz = (
    deployment_db[[DEPLOYMENT_KEY, "Time Zone"]]
    .rename(columns={"Time Zone": "deployment_time_zone"})
)

# --- keep recording id + logger id + deployment id + recording tz ---
rec_tz = (
    recording_db[[RECORDING_KEY, LOGGER_KEY, DEPLOYMENT_KEY, "Time Zone"]]
    .rename(columns={"Time Zone": "recording_time_zone"})
)

# --- merge recording → deployment ---
merged = rec_tz.merge(dep_tz, on=DEPLOYMENT_KEY, how="left")

# --- mismatches ---
merged["timezone_match"] = merged["recording_time_zone"] == merged["deployment_time_zone"]
mismatches = merged[~merged["timezone_match"]].copy()

# --- summary ---
summary = {
    "total_recordings": len(merged),
    "timezone_mismatches": len(mismatches),
    "fraction_mismatched": (len(mismatches) / len(merged)) if len(merged) else 0,
}
print("Summary:")
for k, v in summary.items():
    print(f"  {k}: {v}")

# --- mismatch table (logger id in its own column), sorted as requested ---
mismatch_table = (
    mismatches[[DEPLOYMENT_KEY, RECORDING_KEY, LOGGER_KEY, "recording_time_zone", "deployment_time_zone"]]
    .sort_values([DEPLOYMENT_KEY, RECORDING_KEY])
    .reset_index(drop=True)
)

mismatch_table
